# YOLO workshop: train your own object detector

**What you'll do in this notebook:** take a labeled dataset, teach a computer to find objects in photos, check how good it is, and download **your own model file** that you can use anywhere.

Press **▶ (play)** on each cell, top to bottom, and read the short note above each one. Nothing needs to be typed.

**You'll run this notebook TWICE:**
- **Round 1:** train on the small dataset **you labeled yourself** (`my_dataset.zip` from the labeling website). Notice how well it does.
- **Round 2:** run it again with the big ready-made dataset (`wildlife_dataset.zip`). Compare. *Why is it so different?*

| Step | What happens |
|---|---|
| 1 | Switch on the free GPU |
| 2 | Install YOLO |
| 3 | Load the dataset |
| 4 | Look at the data (always do this!) |
| 5 | **Train** |
| 6 | See how well it learned |
| 7 | Test on your own photo |
| 8 | Download your model |

## 1. Switch on the GPU
Training is heavy maths, and a GPU does it ~20× faster than a normal processor.

**Menu → Runtime → Change runtime type → T4 GPU → Save.** Then run the cell below. It should print a GPU name.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE. Go to Runtime > Change runtime type > T4 GPU")

## 2. Install YOLO
We use **Ultralytics**, the most popular YOLO library. This takes about 30 seconds.

In [ ]:
!pip install -q ultralytics > /dev/null 2>&1
import ultralytics
print("✓ YOLO installed, version", ultralytics.__version__)

## 3. Load the dataset
A dataset is a `.zip` of **photos + labels**. Each photo has a text file saying *what* is in it and *where* (the box).

Run the cell, click **Choose files**, and pick the dataset zip:
- **Round 1:** `my_dataset.zip` (the one you exported from the labeling website)
- **Round 2:** `wildlife_dataset.zip` (from the shared Drive folder)

In [ ]:
#@title Upload your dataset zip
import os, zipfile, glob, shutil

try:
    from google.colab import files
    uploaded = files.upload()          # a "Choose files" button appears
    ZIP = next(iter(uploaded))
except ImportError:                    # running on your own computer instead of Colab
    ZIP = os.environ.get("DATASET_ZIP", "dataset.zip")

shutil.rmtree("dataset", ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall("dataset")

DATA_YAML = glob.glob("dataset/**/data.yaml", recursive=True)[0]
DATA_DIR = os.path.dirname(DATA_YAML)
print("Found:", DATA_YAML)
print(open(DATA_YAML).read())

## 4. Look at the data
**Always look at your data before training.** Most problems with AI models are really problems with the data.

This draws the labeled boxes on a few random photos. The numbers show how many photos are in each split:
- **train**: the photos the model *practices* on
- **val**: photos it never practices on, used to test it fairly

In [ ]:
import random, yaml, matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

names = yaml.safe_load(open(DATA_YAML))["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
colors = ["#3ee6a0", "#ff6b8b", "#5aa9ff", "#ffc24b", "#b78cff", "#3fd5e6"]

for split in ("train", "val"):
    print(split, ":", len(glob.glob(f"{DATA_DIR}/images/{split}/*")), "photos")

def label_path(img):
    return img.replace("/images/", "/labels/").replace("\\images\\", "\\labels\\").rsplit(".", 1)[0] + ".txt"

def show_labeled(n=6):
    imgs = random.sample(glob.glob(f"{DATA_DIR}/images/train/*"), n)
    fig, axs = plt.subplots(2, 3, figsize=(15, 8))
    for ax, p in zip(axs.flat, imgs):
        im = Image.open(p).convert("RGB"); W, H = im.size   # (some datasets are grayscale; RGB keeps them looking normal)
        ax.imshow(im); ax.axis("off")
        if os.path.exists(label_path(p)):
            for line in open(label_path(p)):
                c, x, y, w, h = line.split()[:5]; c = int(c); x, y, w, h = (float(v) for v in (x, y, w, h))
                ax.add_patch(patches.Rectangle(((x - w / 2) * W, (y - h / 2) * H), w * W, h * H, fill=False, lw=2.5, ec=colors[c % 6]))
                ax.text((x - w / 2) * W, (y - h / 2) * H - 4, names[c], color="black", fontsize=9,
                        bbox=dict(fc=colors[c % 6], ec="none", pad=1.5))
    plt.tight_layout(); plt.show()

show_labeled()

## 5. Train 🚀
This is the main event. YOLO starts from a **pretrained model** that already knows how to see edges and shapes, then learns *your* classes on top.

Each **epoch** = one full pass through all the training photos. After every epoch you'll see the mistakes (**loss**) fall and the accuracy (**mAP50**) rise.

You can change these three settings (they're also your levers in the challenge):

| Setting | What it does | Try |
|---|---|---|
| `MODEL` | Brain size: `n` nano < `s` small < `m` medium. Bigger = more accurate but slower. | `yolo11n.pt` |
| `EPOCHS` | How many rounds of practice. Too few = under-trained. | 30 |
| `IMGSZ` | Picture size it looks at. Larger helps small objects. | 640 |

In [ ]:
#@title Settings and training
MODEL = "yolo11n.pt"   #@param ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]
EPOCHS = 30            #@param {type:"integer"}
IMGSZ = 640            #@param {type:"integer"}

from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, project="runs", name="train", exist_ok=True)

RUN_DIR = str(results.save_dir)        # the folder where everything from this training was saved
BEST = os.path.join(RUN_DIR, "weights", "best.pt")
print("\nYour model is saved at:", BEST)

## 6. How well did it learn?
**mAP50** is the standard score: 100% means it finds every object, in the right place, with no false alarms.

Below: the learning curves, a **confusion matrix** (which classes it mixes up), and its guesses on test photos it never practiced on.

In [ ]:
from IPython.display import Image as Show, display

best = YOLO(BEST)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, verbose=False)
print(f"\n>>> mAP50 = {metrics.box.map50:.1%}   (mAP50-95 = {metrics.box.map:.1%})")
print(f">>> precision = {metrics.box.mp:.1%}   recall = {metrics.box.mr:.1%}")

for f in ("results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"):
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p):
        print("\n" + f); display(Show(p, width=900))

## 7. Test it on a new photo
Upload any photo (try one of your own!). The model draws what it finds.

In [ ]:
#@title Upload a photo to test
try:
    from google.colab import files
    up = files.upload()
    TEST = next(iter(up))
except ImportError:
    TEST = os.environ.get("TEST_IMAGE", glob.glob(f"{DATA_DIR}/images/val/*")[0])

res = best.predict(TEST, conf=0.25, save=True, project="runs", name="predict", exist_ok=True)
from collections import Counter
found = Counter(names[int(c)] for c in res[0].boxes.cls)
print("Found", len(res[0].boxes), "objects:", dict(found) if found else "nothing (try a clearer photo)")
# YOLO may rename the saved picture (e.g. photo.jpg.webp -> photo.jpg.jpg), so pick the newest file it saved
saved = max(glob.glob(os.path.join(res[0].save_dir, "*.*")), key=os.path.getmtime)
display(Show(saved, width=700))

## 8. Download your model
`best.pt` is **your trained model**, a single file. It's the version with the best score during training.

You can use it on any computer, in code, or on a live camera:

```bash
pip install ultralytics
yolo predict model=best.pt source=0 show=True      # live webcam
yolo predict model=best.pt source=photo.jpg        # one photo
```

In [ ]:
try:
    from google.colab import files
    files.download(BEST)
except ImportError:
    print("Your model is at:", BEST)

## 🏁 The challenge: make it better
Go back to **step 5** and change **one thing at a time**, then compare the mAP50:
1. A bigger model: `yolo11s.pt`
2. More epochs: `60`
3. A bigger picture size: `800`

Keep notes of what helped. When your team has its best model, download `best.pt` and hand it in. It will be scored on **photos nobody has seen**, so a model that memorized the training photos won't win.

---
### Cheat sheet: the whole thing in 3 commands
```bash
yolo detect train    data=data.yaml  model=yolo11n.pt  epochs=30  imgsz=640    # teach it
yolo detect val      model=best.pt   data=data.yaml                            # score it
yolo detect predict  model=best.pt   source=0  show=True                       # use it
```